# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAliEjaz1/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature vector: one row per page per month, built from the warehouse's
fact table — average position, total impressions, total clicks, and derived
CTR. No categorical encoding needed; all four features are numeric.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"
path = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

fv = con.execute(f"""
    SELECT content_hash_id, client_hash_id,
           AVG(gsc_avg_position) AS avg_position,
           SUM(gsc_impressions) AS total_impressions,
           SUM(gsc_clicks) AS total_clicks
    FROM read_parquet('{path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
fv["ctr"] = np.where(fv["total_impressions"] > 0, fv["total_clicks"] / fv["total_impressions"], 0.0)
print(f"Feature vector: {len(fv)} rows, {fv.isna().sum().sum()} missing values")
fv.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector: 176738 rows, 0 missing values


,content_hash_id,client_hash_id,avg_position,total_impressions,total_clicks,ctr
0,content_05597932fe4da067,client_73cda7b4e4f265ea,2.714744,57.0,0.0,0.000000
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,7.209549,6523.0,7.0,0.001073
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,6.481453,149.0,0.0,0.000000
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2.987198,453.0,0.0,0.000000
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,6.724039,5630.0,6.0,0.001066


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- **avg_position** — mean search rank this month. No missing values (rows
  without GSC data are filtered out). Available same-day: yes.
- **total_impressions** — summed same-month impressions. Available same-day: yes.
- **total_clicks** — summed same-month clicks. Available same-day: yes.
- **ctr** — derived same-month, clicks/impressions with a zero-impression
  guard. Available same-day: yes.

All four exist before the prediction moment — none depend on future data.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
notes = pd.DataFrame({
    "feature": ["avg_position", "total_impressions", "total_clicks", "ctr"],
    "missing_handling": ["rows filtered by gsc_data_available"] * 4,
    "available_before_decision": [True, True, True, True],
})
print(notes.to_string(index=False))

          feature                    missing_handling  available_before_decision
     avg_position rows filtered by gsc_data_available                       True
total_impressions rows filtered by gsc_data_available                       True
     total_clicks rows filtered by gsc_data_available                       True
              ctr rows filtered by gsc_data_available                       True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking a label built from ctr==0 with ctr itself as a feature gives
perfect accuracy (1.000), because the label was built from that exact rule
— confirmed in Week 3. Replacing it with an independent feature
(avg_position) drops accuracy to a realistic 0.595. This is the same test,
repeated here on this feature vector.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fv["is_declining"] = (fv["ctr"] == 0).astype(int)
leaked_pred = (fv["ctr"] == 0).astype(int)
leak_acc = (leaked_pred == fv["is_declining"]).mean()
honest_pred = (fv["avg_position"] > fv["avg_position"].median()).astype(int)
honest_acc = (honest_pred == fv["is_declining"]).mean()
print(f"Leaked feature (ctr==0) accuracy: {leak_acc:.3f}")
print(f"Honest feature (avg_position) accuracy: {honest_acc:.3f}")

Leaked feature (ctr==0) accuracy: 1.000
Honest feature (avg_position) accuracy: 0.571


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **raw query text** — not public-safe, could identify client search terms
- **raw URLs** — not public-safe, could identify clients
- **client_hash_id as a model feature** — used only for grouping in
  validation, never as a predictive input (would let the model memorize
  clients instead of learning generalizable signal)
- **any future-month metric** — would leak the outcome into the features

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = {
    "raw query text": "not public-safe",
    "raw URLs": "not public-safe",
    "client_hash_id as feature": "used for validation grouping only, not prediction",
    "future-month metrics": "would leak the outcome",
}
for k, v in excluded.items():
    print(f"{k}: {v}")

raw query text: not public-safe
raw URLs: not public-safe
client_hash_id as feature: used for validation grouping only, not prediction
future-month metrics: would leak the outcome


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.